# Umba Fraud Detection — EDA & Model Pipeline

This notebook walks through:
1. Data loading and quality checks
2. Exploratory data analysis (class imbalance, missing values, feature distributions)
3. Feature engineering with hypothesis testing
4. Model training with time-series cross-validation
5. Model comparison and selection
6. Calibration and thresholding
7. Predictions on test set


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
identity = pd.read_csv('../data/identity.csv')
print(f'Train: {train.shape}, isFraud: {train.isFraud.mean():.4f}')
print(f'Test: {test.shape}')
print(f'Identity: {identity.shape} (unique TIDs: {identity.TransactionID.nunique()})')


## 1. Data Quality Checks

### Key findings:
- **Class imbalance**: Only 3.44% fraud — need PR-AUC as primary metric
- **`flagged_for_review`**: Correlation of 0.63 with isFraud — **this is leakage!** This field is populated after manual review and would not be available at prediction time. DROPPED from features.
- **Identity join**: Not 1:1 — some TransactionIDs have 2 rows in identity.csv. Need aggregation.
- **Missing data**: High missing rates in dist2 (73%), V5/V10/V20 (65%), D2 (55%), D4-D5 (48-52%)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

train['isFraud'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'crimson'])
axes[0].set_title('Class Distribution')
axes[0].set_xticklabels(['Legitimate', 'Fraud'])

axes[1].bar(['Train', 'Test'], [len(train), len(test)], color=['steelblue', 'gray'])
axes[1].set_title('Dataset Sizes')

leakage_corr = train[['flagged_for_review', 'isFraud']].corr().iloc[0, 1]
axes[2].bar(['flagged_for_review vs isFraud'], [leakage_corr], color='crimson')
axes[2].set_title(f'Leakage Check (corr={leakage_corr:.2f})')
axes[2].axhline(0, color='black', linestyle='-')

plt.tight_layout()
plt.show()

print(f'flagged_for_review non-zero rate: {(train.flagged_for_review > 0).mean():.4f}')
print(f'flagged_for_review correlation with isFraud: {leakage_corr:.4f}')
print(f'\nIdentity max rows per TransactionID: {identity.groupby("TransactionID").size().max()}')


## 2. Missing Data Analysis


In [ ]:
missing = train.isnull().mean().sort_values(ascending=False) * 100
missing = missing[missing > 0]

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(range(len(missing)), missing.values, color='steelblue')
ax.set_yticks(range(len(missing)))
ax.set_yticklabels(missing.index)
ax.set_xlabel('Missing %')
ax.set_title('Missing Values by Feature')
plt.tight_layout()
plt.show()
print(f'Features with >20% missing: {len(missing[missing > 20])} / {len(missing)}')


## 3. Hypothesis Testing for Feature Selection

Before conducting hypothesis tests, we check assumptions:
- **Normality**: Using D'Agostino-Pearson test (large samples). If p > 0.05, data is normal.
- **For normal features**: Welch's t-test (doesn't assume equal variance)
- **For non-normal features**: Mann-Whitney U test (non-parametric)

**Chi-squared test** for categorical features vs fraud status.


In [ ]:
import sys
sys.path.insert(0, '../src')
from hypothesis_tests import hypothesis_test_feature_selection, chi2_feature_selection

numerical_features = ['TransactionAmt', 'dist1', 'dist2', 'recipient_account_age_days',
                      'sender_prev_txn_count'] + [f'C{i}' for i in range(1, 9)] + [f'D{i}' for i in range(1, 6)] + [f'V{i}' for i in range(1, 21)]

X_train = train[numerical_features].copy()
for c in X_train.columns:
    X_train[c] = X_train[c].fillna(-999)

ht_results = hypothesis_test_feature_selection(X_train, train.isFraud, numerical_features)
print("=== Numerical Feature Hypothesis Tests ===")
print(f"Features with significant difference: {(ht_results.reject_null == True).sum()} / {len(ht_results)}")
print()
print(ht_results.head(10))


In [ ]:
categorical_features = ['channel', 'card_type', 'card_bank', 'country', 'currency',
                               'M1', 'M2', 'M3', 'M4', 'M5', 'M6']
chi2_results = chi2_feature_selection(train[categorical_features].fillna('missing'), train.isFraud, categorical_features)
print("=== Categorical Feature Chi-Squared Tests ===")
print(chi2_results)


## 4. Feature Engineering

Key steps:
- Drop `flagged_for_review` (leakage)
- Aggregate identity.csv (mode for DeviceType/DeviceInfo, mean for id_*)
- Log transform TransactionAmt
- Create amount deviation from currency mean
- One-hot encode card_type and channel
- Frequency encoding for high-cardinality categoricals
- Missing value indicators for V features
- Email domain extraction
- Account age binning
- Interaction: amount x channel, amount x sender_txn_count


In [ ]:
from preprocessing import load_data, preprocess
train, test, identity = load_data('../data')
X_train, X_test, y = preprocess(train, test, identity)

print(f'Final feature matrix: {X_train.shape}')
print(f'Features created: {X_train.shape[1]}')
print(f'\nFeature types:')
print(X_train.dtypes.value_counts())


## 5. Model Training — Time-Series Cross-Validation

Using `TimeSeriesSplit` (5 folds) because test data is later in time than training data — simulating production conditions.

Models compared:
1. **Logistic Regression** (baseline, interpretable)
2. **Random Forest** (ensemble, handles non-linearity)
3. **XGBoost** (gradient boosting)
4. **LightGBM** (gradient boosting, faster)

**Primary metric**: PR-AUC (honest under class imbalance)
**Secondary**: ROC-AUC, Brier score (calibration)


In [ ]:
from model_training import run_model_comparison

final_model, test_probs, cv_results, best_model = run_model_comparison(X_train, X_test, y)

print(f'\n=== Final Model: {best_model} ===')
print(f'PR-AUC: {cv_results[best_model]["mean_PR_AUC"]:.4f}')


## 6. Results & Interpretation


In [ ]:
results_df = pd.DataFrame([
    {'Model': k, 'PR-AUC': v['mean_PR_AUC'], 'PR-AUC Std': v['std_PR_AUC'],
     'ROC-AUC': v['mean_ROC_AUC'], 'Brier': v['mean_Brier']}
    for k, v in cv_results.items()
])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, metric in enumerate(['PR-AUC', 'ROC-AUC', 'Brier']):
    colors = ['crimson' if r['Model'] == best_model else 'steelblue' for _, r in results_df.iterrows()]
    axes[i].bar(results_df['Model'], results_df[metric], color=colors)
    axes[i].set_title(f'{metric} Comparison')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()
print(results_df.to_string(index=False))


In [ ]:
print("=== Test Set Predictions ===")
submission = pd.DataFrame({
    'TransactionID': test['TransactionID'].values,
    'isFraud_prob': test_probs
})
submission.to_csv('../predictions.csv', index=False)
print(f'Shape: {submission.shape}')
print(f'Fraud rate: {test_probs.mean():.4f}')
print(f'Range: [{test_probs.min():.4f}, {test_probs.max():.4f}]')
print(f'\nFirst 10 predictions:')
print(submission.head(10))


## 7. Conclusions & Next Steps

### Achieved:
- Cleaned data pipeline handling identity join, missing values, leakage
- Time-series cross-validation (production-realistic)
- 4 model comparison with PR-AUC metric
- Calibration via isotonic regression
- `predictions.csv` generated

### What I'd improve with more time:
1. **Feature engineering**: Create more aggregate features (rolling stats per card/user)
2. **Hyperparameter tuning**: Grid/Random search for best model
3. **Anomaly detection**: Add unsupervised features (isolation forest scores)
4. **Ensemble**: Stack models for better performance
5. **Threshold optimization**: Choose threshold based on cost of false positives vs false negatives
6. **Model monitoring**: Drift detection, retraining pipeline
7. **SHAP analysis**: Model interpretability for ops team
